# Practice: MLOps
*AI from scratch, stage 7* · lesson: https://ihor.sh/ai/mlops.html

The operations side of the outage model: a drift score (PSI) between windows, a data validator that stops a broken
batch, a champion-versus-challenger comparison on the newest days with a promotion gate, a daily prediction monitor,
and a model card. Runs on a CPU in seconds. Replace each `...` under `# YOUR CODE HERE`, run the cell, then the check
under it; `hint(n)` and `answer(n)` are there when you're stuck.

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['p, q = p / p.sum(), q / q.sum()', 'return float(np.sum((q - p) * np.log(q / p)))'], 2: ['weekday = f[f["start"].dt.dayofweek < 5]; weekend = f[f["start"].dt.dayofweek >= 5]', 'reason_psi = psi(weekday["reason"], weekend["reason"], cats)'], 3: ['if not batch["kind"].isin(["EL", "ES"]).all(): problems.append("unknown kind")', 'if batch[FEATURES].isna().any().any(): problems.append("missing values")'], 4: ['auc_champ = roc_auc_score(late["hours"] > 4, champion.predict_proba(late[FEATURES])[:, 1]); the same for challenger', 'promote = auc_chall > auc_champ + 0.01'], 5: ['probs = champion.predict_proba(late[FEATURES])[:, 1]', 'daily = pd.Series(probs, index=late.index).groupby(late["start"].dt.date).mean()'], 6: ['card = {"intended_use": "...", "training_data": "...", "evaluation": f"AUC {auc_champ:.2f} on ...", "limitations": "..."}']}
_ANSWERS = {1: 'cCwgcSA9IHAgLyBwLnN1bSgpLCBxIC8gcS5zdW0oKQpyZXR1cm4gZmxvYXQobnAuc3VtKChxIC0gcCkgKiBucC5sb2cocSAvIHApKSk=', 2: 'd2Vla2RheSA9IGZbZlsic3RhcnQiXS5kdC5kYXlvZndlZWsgPCA1XQp3ZWVrZW5kID0gZltmWyJzdGFydCJdLmR0LmRheW9md2VlayA+PSA1XQpyZWFzb25fcHNpID0gcHNpKHdlZWtkYXlbInJlYXNvbiJdLCB3ZWVrZW5kWyJyZWFzb24iXSwgY2F0cyk=', 3: 'aWYgbm90IGJhdGNoWyJraW5kIl0uaXNpbihbIkVMIiwgIkVTIl0pLmFsbCgpOgogICAgcHJvYmxlbXMuYXBwZW5kKCJ1bmtub3duIGtpbmQiKQppZiBiYXRjaFtGRUFUVVJFU10uaXNuYSgpLmFueSgpLmFueSgpOgogICAgcHJvYmxlbXMuYXBwZW5kKCJtaXNzaW5nIHZhbHVlcyIp', 4: 'YXVjX2NoYW1wID0gcm9jX2F1Y19zY29yZShsYXRlWyJob3VycyJdID4gNCwgY2hhbXBpb24ucHJlZGljdF9wcm9iYShsYXRlW0ZFQVRVUkVTXSlbOiwgMV0pCmF1Y19jaGFsbCA9IHJvY19hdWNfc2NvcmUobGF0ZVsiaG91cnMiXSA+IDQsIGNoYWxsZW5nZXIucHJlZGljdF9wcm9iYShsYXRlW0ZFQVRVUkVTXSlbOiwgMV0pCnByb21vdGUgPSBhdWNfY2hhbGwgPiBhdWNfY2hhbXAgKyAwLjAx', 5: 'cHJvYnMgPSBjaGFtcGlvbi5wcmVkaWN0X3Byb2JhKGxhdGVbRkVBVFVSRVNdKVs6LCAxXQpkYWlseSA9IHBkLlNlcmllcyhwcm9icywgaW5kZXg9bGF0ZS5pbmRleCkuZ3JvdXBieShsYXRlWyJzdGFydCJdLmR0LmRhdGUpLm1lYW4oKQ==', 6: 'Y2FyZCA9IHsiaW50ZW5kZWRfdXNlIjogIldhcm4gcmlkZXJzIHdoZW4gYW4gZWxldmF0b3Igb3IgZXNjYWxhdG9yIG91dGFnZSBpcyBsaWtlbHkgdG8gbGFzdCBtb3JlIHRoYW4gNCBob3Vycy4iLAogICAgICAgICJ0cmFpbmluZ19kYXRhIjogZiJ7bGVuKGVhcmx5KX0gZmluaXNoZWQgTllDIHN1YndheSBvdXRhZ2VzIGZyb20gdGhlIGlob3Iuc2ggTVRBIGFyY2hpdmUsIGJlZm9yZSAyMDI2LTEwLTEyLiIsCiAgICAgICAgImV2YWx1YXRpb24iOiBmIkFVQyB7YXVjX2NoYW1wOi4yZn0gb24ge2xlbihsYXRlKX0gb3V0YWdlcyBmcm9tIDIwMjYtMTAtMTIgb24gKHRpbWUgc3BsaXQpLiIsCiAgICAgICAgImxpbWl0YXRpb25zIjogIk9uZSB3ZWVrIG9mIGRhdGE6IHNlYXNvbnMsIGhvbGlkYXlzIGFuZCBuZXcga2luZHMgb2Ygb3V0YWdlIGFyZSB1bnNlZW47IHdlZWtlbmQgcmVhc29ucyBkaWZmZXIgZnJvbSB3ZWVrZGF5cy4ifQ=='}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

df = pd.read_csv("https://raw.githubusercontent.com/ikruchynskyi/ihor.sh/main/ai/data/outages.csv", parse_dates=["start", "last_seen"])
f = df[df["finished"]].copy()
f["hours"] = (f["last_seen"] - f["start"]).dt.total_seconds() / 3600
f["hour"] = f["start"].dt.hour; f["n_lines"] = f["lines"].str.count("/") + 1
FEATURES = ["kind", "reason", "ada", "hour", "n_lines"]
print(len(f), "finished outages,", f["start"].min().date(), "to", f["start"].max().date())

## 1. The population stability index
`psi(ref, live, cats)` starts by computing each category's share in `ref` (`p`) and in `live` (`q`), plus a small
floor so no share is 0. Finish it: renormalize both to add up to 1, and return `sum((q - p) * ln(q / p))`.

In [ ]:
def psi(ref, live, cats, floor=1e-3):
    p = np.array([(ref == c).mean() for c in cats]) + floor
    q = np.array([(live == c).mean() for c in cats]) + floor
    # YOUR CODE HERE (stuck? hint(1), then answer(1))
    ...

In [ ]:
s = pd.Series(["a", "b", "a", "b"])
check(1, lambda: abs(psi(s, s, ["a", "b"])) < 1e-12 and psi(s, pd.Series(["a"] * 4), ["a", "b"]) > 1,
      "0 for identical windows, large when a category vanishes")

## 2. Weekdays against weekends
Set `weekday` and `weekend` to the outages starting Monday–Friday and Saturday–Sunday (`f["start"].dt.dayofweek` is
0 for Monday … 6 for Sunday), then `reason_psi`: the PSI of `"reason"` between them over all reasons.

In [ ]:
cats = sorted(f["reason"].unique())
weekday = weekend = reason_psi = None
# YOUR CODE HERE (stuck? hint(2), then answer(2))
...
print("reason PSI, weekdays vs weekend:", reason_psi)

In [ ]:
check(2, lambda: abs(reason_psi - 0.324) < 0.01, "about 0.32, over the 0.25 'real shift' line: and it's just the city's weekly rhythm")

## 3. Stop a broken batch before it trains anything
`validate(batch)` returns a list of problems (strings), empty when the batch is fine. The column check and the
hour check are written. Add two more in the same style: `kind` must be only `"EL"` or `"ES"` (`batch["kind"].isin([...])`
is True where a value is in the list), and no feature may be missing (`batch[FEATURES].isna()` is True where a value is
missing; `.any().any()` asks "anywhere?"). The broken batch below (given) has three problems.

In [ ]:
def validate(batch):
    problems = []
    missing = [c for c in FEATURES if c not in batch]
    if missing:
        return [f"missing columns: {missing}"]
    if not batch["hour"].between(0, 23).all():
        problems.append("hour out of range")
    # YOUR CODE HERE (stuck? hint(3), then answer(3))
    ...
    return problems

broken = f[FEATURES].head(50).copy()
broken.loc[broken.index[:5], "hour"] = broken["hour"].head(5) * 60          # someone sent minutes instead of hours
broken.loc[broken.index[7], "kind"] = "Elevator"
broken.loc[broken.index[9], "reason"] = None
print("clean batch:", validate(f[FEATURES]), "· broken batch:", validate(broken))

In [ ]:
check(3, lambda: validate(f[FEATURES]) == [] and len(validate(broken)) == 3 and validate(broken.drop(columns="hour")) != [],
      "nothing wrong with the real data; all three problems found in the broken batch; a missing column is caught first")

## 4. Champion against challenger, on the newest days
Both are trained on outages before October 12 and scored on the ones from then on (the time split of toolkit 2). The
champion is the logistic regression; the challenger, boosted trees. Set `auc_champ` and `auc_chall`, then `promote`:
`True` only if the challenger beats the champion by more than 0.01 (smaller gaps are noise at this size).

In [ ]:
early, late = f[f["start"] < "2026-10-12"], f[f["start"] >= "2026-10-12"]
prep = lambda: make_column_transformer((OneHotEncoder(handle_unknown="ignore"), ["kind", "reason"]), (StandardScaler(), ["ada", "hour", "n_lines"]))
champion = make_pipeline(prep(), LogisticRegression(max_iter=2000)).fit(early[FEATURES], early["hours"] > 4)
challenger = make_pipeline(prep(), HistGradientBoostingClassifier(random_state=0)).fit(early[FEATURES], early["hours"] > 4)
auc_champ = auc_chall = promote = None
# YOUR CODE HERE (stuck? hint(4), then answer(4))
...
print("champion AUC", auc_champ, "· challenger AUC", auc_chall, "· promote:", promote)

In [ ]:
check(4, lambda: 0.85 < auc_champ < 0.95 and promote == (auc_chall > auc_champ + 0.01),
      "the champion holds about 0.90 on the newest days; promotion follows the rule, not a hunch")

## 5. Watch the outputs every day
Labels arrive late, predictions don't. Set `daily`: the champion's mean predicted probability of a long outage for
each start date of the late window (add a column of predictions, then group by `late["start"].dt.date`).

In [ ]:
daily = None
# YOUR CODE HERE (stuck? hint(5), then answer(5))
...
print(daily.round(3) if daily is not None else None)

In [ ]:
check(5, lambda: len(daily) >= 3 and daily.between(0, 1).all(), "one average per day: a jump here would be the earliest warning, before any outage has even ended")

## 6. A model card
Fill `card`, a dict with the keys `"intended_use"`, `"training_data"`, `"evaluation"` and `"limitations"`, each a short
sentence of your own. The evaluation should mention the champion's AUC on the newest days.

In [ ]:
card = None
# YOUR CODE HERE (stuck? hint(6), then answer(6))
...
for k, v in (card or {}).items():
    print(f"{k}: {v}")

In [ ]:
check(6, lambda: set(card) == {"intended_use", "training_data", "evaluation", "limitations"} and all(len(v) > 20 for v in card.values()) and "AUC" in card["evaluation"],
      "four short answers a future maintainer will be glad to find")

## Next
Back to the path: https://ihor.sh/ai/ · Google, "Rules of Machine Learning": https://developers.google.com/machine-learning/guides/rules-of-ml